# Bhagvad Mood Journal - PyTorch Machine Learning Pipeline

This notebook demonstrates how the machine learning pipeline works in the **Bhagvad Mood Journal** application. We transitioned the machine learning code from TensorFlow/Keras to **PyTorch**.

We will cover:
1. Setup and imports
2. Preparing datasets (synthetic and structured text)
3. Defining PyTorch Neural Networks
4. Training loops, losses, and optimizers
5. Evaluating models for resource recommendations, crisis detection, and mood classification

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Determine hardware acceleration device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Wellness Resource Recommendation Model

The wellness recommendation model takes $5$ normalized inputs:
- **Age** (normalized: $age / 100$)
- **Gender** (female: $0.0$, male: $0.5$, other/not-specified: $1.0$)
- **PHQ-9** depression screening score (normalized: $score / 27$)
- **GAD-7** anxiety screening score (normalized: $score / 21$)
- **Journal Sentiment Polarity** (normalized: $(polarity + 1) / 2$)

And predicts $4$ categories of resources:
- `0`: Self-Care & Info Guides
- `1`: Stress Management Videos
- `2`: Counselor Booking / Consultation
- `3`: Immediate Crisis Intervention

In [ ]:
def generate_synthetic_data(num_samples=400):
    X = []
    y = []
    for _ in range(num_samples):
        age = np.random.randint(15, 80)
        gender_val = np.random.choice([0.0, 0.5, 1.0])
        phq = np.random.randint(0, 28)
        gad = np.random.randint(0, 22)
        polarity = np.random.uniform(-1.0, 1.0)
        
        age_norm = age / 100.0
        phq_norm = phq / 27.0
        gad_norm = gad / 21.0
        pol_norm = (polarity + 1.0) / 2.0
        
        # Simple labeling rules mapping metrics to appropriate support levels
        if phq >= 15 or gad >= 15 or (phq >= 10 and polarity < -0.6):
            label = 3  # Crisis
        elif polarity < -0.3 and gad >= 8:
            label = 2  # Counselor Booking
        elif age <= 25 and gad >= 7:
            label = 1  # Stress Videos (Young adult)
        else:
            label = 0  # Info Guides
            
        X.append([age_norm, gender_val, phq_norm, gad_norm, pol_norm])
        one_hot = [0.0] * 4
        one_hot[label] = 1.0
        y.append(one_hot)
        
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_np, y_np = generate_synthetic_data(500)
print(f"Data shapes: Features = {X_np.shape}, Targets = {y_np.shape}")

### Defining the PyTorch Model
We create a Sequential feedforward neural network in PyTorch: `5` inputs $\rightarrow$ `8` hidden units $\rightarrow$ `4` output classes with Softmax activation.

In [ ]:
class WellnessPyTorchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 8)
        self.fc2 = nn.Linear(8, 4)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.softmax(self.fc2(x), dim=-1)
        return x

model = WellnessPyTorchModel().to(device)
print(model)

### Training the Recommendation Model

In [ ]:
# Set up DataLoader
inputs = torch.tensor(X_np, dtype=torch.float32).to(device)
targets = torch.tensor(y_np, dtype=torch.float32).to(device)

dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Starting Training...")
model.train()
for epoch in range(100):
    epoch_loss = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        # Categorical crossentropy loss: -mean(sum(y_true * log(y_pred)))
        loss = -torch.mean(torch.sum(batch_y * torch.log(preds + 1e-15), dim=-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | Avg Loss = {epoch_loss / len(X_np):.5f}")
print("Training completed!")

## 2. NLP Text Classifiers

We have two separate classifiers for student journal thoughts:
1. **Crisis Detection Model**: Predicts whether a thought indicates a crisis (`1` for crisis, `0` for safe/normal).
2. **Mood Classification Model**: Predicts one of 6 mood states (`happy`, `sad`, `anxious`, `angry`, `neutral`, `overwhelmed`).

Both models convert text into binary presence vectors of a vocabulary size of $75$ wellness/crisis terms.

In [ ]:
VOCABULARY = [
    # Crisis indicators
    "suicide", "suicidal", "kill", "die", "death", "overdose", "harm", "cut", "bleed",
    "stroke", "choke", "accident", "chest", "breathe", "emergency", "poison", "heart attack",
    "end my life", "ending it", "cannot go on", "myself", "life",
    # Anxious
    "anxious", "worry", "panic", "scared", "fear", "nervous", "tense", "stress", "restless",
    # Sad
    "sad", "depressed", "cry", "lonely", "hopeless", "useless", "worthless", "hurt", "grief",
    "devastated", "heartbroken",
    # Angry
    "angry", "mad", "furious", "hate", "irritate", "annoy", "frustrate", "rage",
    # Happy/Positive
    "happy", "good", "great", "glad", "awesome", "excited", "content", "peaceful", "grateful", "hopeful",
    # Neutral
    "fine", "okay", "neutral", "normal", "average", "meh",
    # Stressed/Overwhelmed
    "overwhelmed", "exhausted", "burned out", "burn out"
]

def text_to_vector(text):
    text_lower = text.lower()
    vector = np.zeros(len(VOCABULARY), dtype=np.float32)
    for idx, word in enumerate(VOCABULARY):
        if word in text_lower:
            vector[idx] = 1.0
    return vector

# Example vector mapping
text = "I am so happy but feeling burned out and anxious"
vec = text_to_vector(text)
print(f"Vocab size: {len(VOCABULARY)}")
print(f"Non-zero indices in vector: {np.where(vec > 0)[0]}")

### Defining the Text Classifier Architecture
The NLP models use a neural network with `input_dim` inputs $\rightarrow$ `12` hidden units $\rightarrow$ `output_dim` outputs.

In [ ]:
class TextPyTorchModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 12)
        self.fc2 = nn.Linear(12, output_dim)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.softmax(self.fc2(x), dim=-1)
        return x

crisis_model = TextPyTorchModel(len(VOCABULARY), 2).to(device)
mood_model = TextPyTorchModel(len(VOCABULARY), 6).to(device)
print(crisis_model)

### Training the NLP Models

In [ ]:
CRISIS_TRAINING_PAIRS = [
    ("I want to end my life", 1), ("I am thinking of suicide", 1), ("I want to kill myself", 1),
    ("I will overdose on pills", 1), ("I want to die today", 1), ("cannot go on anymore, ending it", 1),
    ("He is having a sudden stroke and collapsed", 1), ("chest pain and cannot breathe", 1),
    ("Having a heart attack, call an ambulance", 1), ("Had a major car accident, bleeding", 1),
    ("I am feeling a bit sad today", 0), ("I am anxious about my exam", 0), ("It was a normal, okay day", 0),
    ("I am so angry at my brother", 0), ("We had a great time walking", 0), ("I am happy and content", 0)
]

MOOD_TRAINING_PAIRS = [
    ("I am very happy and joyful", 0), ("It is a great day, I feel content", 0), ("I am peaceful, calm and relaxed", 0),
    ("I am feeling very sad and down", 1), ("I feel depressed and lonely", 1), ("I want to cry, empty", 1),
    ("I am feeling extremely anxious and nervous", 2), ("scared and worried about the outcome", 2), ("My panic is rising", 2),
    ("I am so angry and mad", 3), ("I hate this situation, furious", 3), ("irritated and annoyed", 3),
    ("I feel okay and fine", 4), ("Just a normal day, average", 4), ("meh, nothing special", 4),
    ("I am overwhelmed by too much work", 5), ("stressed out and exhausted", 5), ("burned out and swamped", 5)
]

# Create Vectors
X_c, y_c = [], []
for text, label in CRISIS_TRAINING_PAIRS:
    X_c.append(text_to_vector(text))
    one_hot = [0.0, 0.0]
    one_hot[label] = 1.0
    y_c.append(one_hot)
X_c, y_c = np.array(X_c, dtype=np.float32), np.array(y_c, dtype=np.float32)

X_m, y_m = [], []
for text, label in MOOD_TRAINING_PAIRS:
    X_m.append(text_to_vector(text))
    one_hot = [0.0] * 6
    one_hot[label] = 1.0
    y_m.append(one_hot)
X_m, y_m = np.array(X_m, dtype=np.float32), np.array(y_m, dtype=np.float32)

# Train function
def train_nlp_model(model, X, y, epochs=250, batch_size=8):
    inputs = torch.tensor(X, dtype=torch.float32).to(device)
    targets = torch.tensor(y, dtype=torch.float32).to(device)
    dataset = TensorDataset(inputs, targets)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    model.train()
    for epoch in range(epochs):
        for bx, by in loader:
            optimizer.zero_grad()
            preds = model(bx)
            loss = -torch.mean(torch.sum(by * torch.log(preds + 1e-15), dim=-1))
            loss.backward()
            optimizer.step()

print("Training Crisis Detection Model...")
train_nlp_model(crisis_model, X_c, y_c, epochs=250)
print("Training Mood Classification Model...")
train_nlp_model(mood_model, X_m, y_m, epochs=250)
print("NLP training completed!")

## 3. Inference and Verification

Let's test our trained PyTorch models with test scenarios.

In [ ]:
print("--- TEST CRISIS DETECTION ---")
test_phrases = [
    "I feel so happy that I want to cry of joy",
    "I want to end my life, please help me",
    "Just going to the market to buy books",
    "Had a severe chest pain and collapsed"
]

crisis_model.eval()
with torch.no_grad():
    for phrase in test_phrases:
        vec = text_to_vector(phrase)
        t_in = torch.tensor([vec], dtype=torch.float32).to(device)
        probs = crisis_model(t_in).cpu().numpy()[0]
        is_crisis = probs[1] >= 0.5
        status = "[CRISIS]" if is_crisis else "[SAFE]"
        print(f"{status:<9} | Phrase: '{phrase}' | Prob = {probs[1]:.4f}")

print("\n--- TEST MOOD CLASSIFICATION ---")
idx_to_mood = {0: "happy", 1: "sad", 2: "anxious", 3: "angry", 4: "neutral", 5: "overwhelmed"}
mood_phrases = [
    "I feel extremely excited about this design!",
    "I am so lonely and everything is dark.",
    "Worry and panic are taking over my brain.",
    "This makes me so mad and furious!",
    "It is a normal, average day, doing okay.",
    "So much homework, I am overwhelmed and burned out!"
]

mood_model.eval()
with torch.no_grad():
    for phrase in mood_phrases:
        vec = text_to_vector(phrase)
        t_in = torch.tensor([vec], dtype=torch.float32).to(device)
        probs = mood_model(t_in).cpu().numpy()[0]
        mood_idx = np.argmax(probs)
        print(f"Predicted: {idx_to_mood[mood_idx]:<12} | Phrase: '{phrase}' | Prob = {probs[mood_idx]:.4f}")